![Banner Votações](https://www.camara.leg.br/internet/deputado/img/logo-camara.png)

# 🗳️ Fast Track — 04. Bronze: Votações

**Ingestão de Dados Brutos — API Câmara dos Deputados**

| Item | Descrição |
|---|---|
| **Objetivo** | Ingerir dados brutos de votações da API da Câmara dos Deputados para a camada Bronze |
| **Entidade** | Votações Nominais (votações com registro individual de cada voto) |
| **Endpoint API** | `https://dadosabertos.camara.leg.br/api/v2/votacoes` |
| **Tabela Destino** | `uc_fast_track.bronze.votacoes_raw` |
| **Modo** | APPEND (histórico completo de todas as execuções) |
| **Checkpoint** | Por `id` da votação (último ID processado) |
| **Formato Bronze** | Payload JSON raw (STRING) + colunas técnicas |
| **Responsável** | Ernesto Bassoli |
| **Programa** | Upskill Tiller – Engenharia de Dados (T2.7) |

---

## 📋 O Que Este Notebook Faz

1. **Recupera o Load ID** do notebook 00_init_run
2. **Busca checkpoint** (último ID processado)
3. **Chama API** com paginação
4. **Salva JSON raw** em `bronze.votacoes_raw`
5. **Registra logs** completos
6. **Atualiza checkpoint**
7. **Valida** dados ingeridos

---

## 🎯 O Que São Votações?

**Votações Nominais** são aquelas em que o **voto de cada deputado** é registrado individualmente:

**Características:**
* Cada deputado vota: Sim, Não, Abstenção, Obstrução
* Associadas a proposições (projetos de lei, emendas)
* Ocorrem em sessões plenárias ou comissões
* Resultado: Aprovada/Rejeitada
* Essenciais para análise de **comportamento parlamentar** e **disciplina partidária**

---

## 🔄 Quando Executar

* **Workflow A (Diário)**: Task A04_bronze_votacoes
* **Workflow B (Micro-batch)**: Task B04_bronze_votacoes
* **Manual**: Reprocessamento ou testes

## ⚙️ Bloco 1: Configuração e Load ID

Recupera variáveis do notebook `00_init_run`.

In [0]:
print("🔗 Recuperando variáveis...")
print()

try:
    print(f"✅ Load ID: {load_id}")
except NameError:
    raise Exception("Execute notebook 00 primeiro")

try:
    print(f"✅ Ambiente: {env}")
    print(f"✅ Run Date: {run_date}")
    print(f"✅ Full Refresh: {full_refresh}")
    print(f"✅ Catálogo: {catalog}")
except NameError as e:
    raise Exception(f"Variável não encontrada: {e}")

entity_name = "votacoes"
api_base_url = "https://dadosabertos.camara.leg.br/api/v2"
api_endpoint = f"{api_base_url}/votacoes"
table_bronze = f"{catalog}.{schema_bronze}.{entity_name}_raw"
page_size = 100
max_retries = 3
retry_delay_seconds = 5

print(f"\n✅ Entidade: {entity_name}")
print(f"✅ Endpoint: {api_endpoint}")
print(f"✅ Tabela: {table_bronze}")
print("="*70)

## 📦 Bloco 2: Importações

In [0]:
print("📦 Importando bibliotecas...")

import requests
import json
import time
import uuid
import hashlib
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType
from pyspark.sql.functions import col, lit

print("✅ Bibliotecas OK")

## 🔌 Bloco 3: Função de Extração

Define `get_votacoes_from_api()` com paginação e retry.

In [0]:
def get_votacoes_from_api(checkpoint_id=None):
    print(f"🔌 Extraindo de: {api_endpoint}")
    all_data = []
    pagina = 1
    has_more = True
    total_reqs = 0
    
    while has_more:
        print(f"📄 Página {pagina}...")
        params = {"pagina": pagina, "itens": page_size, "ordem": "ASC", "ordenarPor": "id"}
        req_id = str(uuid.uuid4())
        start = time.time()
        success = False
        resp_data = None
        status = None
        err = None
        
        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(api_endpoint, params=params, timeout=30)
                status = r.status_code
                if r.status_code == 200:
                    resp_data = r.json()
                    success = True
                    break
                else:
                    err = f"HTTP {r.status_code}"
                    print(f"   ⚠️ Tentativa {attempt}/{max_retries}: {err}")
                    if attempt < max_retries:
                        time.sleep(retry_delay_seconds * attempt)
            except Exception as e:
                err = str(e)
                print(f"   ⚠️ Tentativa {attempt}/{max_retries}: {err}")
                if attempt < max_retries:
                    time.sleep(retry_delay_seconds * attempt)
        
        resp_time = int((time.time() - start) * 1000)
        total_reqs += 1
        
        spark.createDataFrame([{
            "request_id": req_id, "load_id": load_id, "entity_name": entity_name,
            "endpoint": f"{api_endpoint}?pagina={pagina}", "http_method": "GET",
            "http_status": status, "response_time_ms": resp_time, "success": success,
            "error_message": err, "request_timestamp": datetime.now()
        }]).write.mode("append").saveAsTable(f"{catalog}.{schema_ops}.ingestion_requests")
        
        if not success:
            raise Exception(f"Falha após {max_retries} tentativas: {err}")
        
        data_page = resp_data.get("dados", [])
        print(f"   ✅ {len(data_page)} registros ({resp_time}ms)")
        
        if checkpoint_id:
            data_page = [d for d in data_page if d.get("id", 0) > checkpoint_id]
            print(f"   🔍 Filtrados: {len(data_page)} novos")
        
        all_data.extend(data_page)
        links = resp_data.get("links", [])
        has_more = any(l.get("rel") == "next" for l in links) and len(data_page) > 0
        if has_more:
            pagina += 1
            time.sleep(0.5)
        else:
            print("   ℹ️ Última página")
        print()
    
    print(f"✅ Extração completa: {total_reqs} requests, {len(all_data)} registros")
    return all_data

print("✅ Função definida")

## 🔖 Bloco 4: Checkpoint

Busca último ID processado.

In [0]:
print("🔖 Buscando checkpoint...\n")

if full_refresh:
    print("🔄 FULL REFRESH - ignorando checkpoint")
    checkpoint_value = None
else:
    print("📈 INCREMENTAL")
    try:
        df_ck = spark.sql(f"""
            SELECT checkpoint_value, updated_at
            FROM {catalog}.{schema_ops}.checkpoints
            WHERE source_endpoint = '/votacoes' AND checkpoint_type = 'last_id'
            ORDER BY updated_at DESC LIMIT 1
        """)
        if df_ck.count() > 0:
            row = df_ck.first()
            checkpoint_value = int(row["checkpoint_value"])
            print(f"   ✅ Checkpoint: {checkpoint_value}")
        else:
            print("   ℹ️ Primeira execução")
            checkpoint_value = None
    except:
        checkpoint_value = None

print(f"\n📌 Checkpoint: {checkpoint_value}")
print("="*70)

## 🗄️ Bloco 5: Criar Tabela Bronze

In [0]:
print(f"🗄️ Criando {table_bronze}...\n")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_bronze} (
    payload STRING COMMENT 'Payload JSON raw',
    record_id STRING COMMENT 'ID da votação',
    record_hash STRING COMMENT 'Hash SHA256',
    load_id STRING COMMENT 'UUID da execução',
    ingestion_timestamp TIMESTAMP COMMENT 'Timestamp ingestão',
    ingestion_date DATE COMMENT 'Data ingestão',
    source_endpoint STRING COMMENT 'Endpoint API',
    env STRING COMMENT 'Ambiente'
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze - Votações Raw'
""")

print("✅ Tabela criada/validada")
print("="*70)

## 📥 Bloco 6: Ingestão e Enriquecimento

In [0]:
print("📥 Iniciando ingestão...\n")
ingest_start = time.time()

data_list = get_votacoes_from_api(checkpoint_id=checkpoint_value)
num_records_extracted = len(data_list)

print(f"✅ {num_records_extracted} registros extraídos\n")

if num_records_extracted == 0:
    print("⚠️ Sem dados novos")
    spark.createDataFrame([{
        "load_id": load_id, "entity_name": entity_name, "records_extracted": 0,
        "records_ingested": 0, "api_calls_count": 0,
        "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
        "checkpoint_after": str(checkpoint_value) if checkpoint_value else None,
        "execution_time_sec": int(time.time() - ingest_start), "status": "NO_NEW_DATA",
        "run_date": run_date, "env": env, "created_at": datetime.now()
    }]).write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")
    dbutils.notebook.exit("NO_NEW_DATA")

print("🔧 Enriquecendo...")
enriched = []
ts = datetime.now()
dt = ts.date()

for item in data_list:
    pjson = json.dumps(item, ensure_ascii=False)
    phash = hashlib.sha256(pjson.encode('utf-8')).hexdigest()
    rid = str(item.get("id", "unknown"))
    enriched.append({
        "payload": pjson, "record_id": rid, "record_hash": phash,
        "load_id": load_id, "ingestion_timestamp": ts, "ingestion_date": dt,
        "source_endpoint": "/votacoes", "env": env
    })

print(f"✅ {len(enriched)} enriquecidos\n")

schema = StructType([
    StructField("payload", StringType()), StructField("record_id", StringType()),
    StructField("record_hash", StringType()), StructField("load_id", StringType()),
    StructField("ingestion_timestamp", TimestampType()), StructField("ingestion_date", DateType()),
    StructField("source_endpoint", StringType()), StructField("env", StringType())
])

df_bronze = spark.createDataFrame(enriched, schema=schema)
print(f"✅ DataFrame: {df_bronze.count()} registros")
print("="*70)

## 💾 Bloco 7: Salvar (APPEND)

In [0]:
print(f"💾 Salvando em {table_bronze}...\n")
write_start = time.time()

try:
    df_bronze.write.format("delta").mode("append").partitionBy("ingestion_date").saveAsTable(table_bronze)
    write_time = int(time.time() - write_start)
    print(f"✅ Salvos: {num_records_extracted} registros ({write_time}s)")
    ingest_success = True
    ingest_error = None
except Exception as e:
    ingest_success = False
    ingest_error = str(e)
    print(f"❌ ERRO: {e}")
    raise
finally:
    ingest_total = int(time.time() - ingest_start)
    print(f"⏱️ Tempo total: {ingest_total}s")
    print("="*70)

## 🔖 Bloco 8: Atualizar Checkpoint

In [0]:
print("🔖 Atualizando checkpoint...\n")

max_id = df_bronze.selectExpr("CAST(record_id AS INT) as id_int").agg({"id_int": "max"}).first()[0]

print(f"📊 Anterior: {checkpoint_value}, Novo: {max_id}\n")

if max_id:
    spark.sql(f"""
        MERGE INTO {catalog}.{schema_ops}.checkpoints AS target
        USING (SELECT '/votacoes' AS source_endpoint, 'last_id' AS checkpoint_type,
                      '{max_id}' AS checkpoint_value, current_timestamp() AS updated_at) AS source
        ON target.source_endpoint = source.source_endpoint AND target.checkpoint_type = source.checkpoint_type
        WHEN MATCHED THEN UPDATE SET target.checkpoint_value = source.checkpoint_value, target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Checkpoint atualizado: {max_id}")
else:
    print("⚠️ Sem ID válido")

print("="*70)

## 📝 Bloco 9: Logging

In [0]:
print("📝 Registrando log...\n")

api_calls = spark.sql(f"""
    SELECT COUNT(*) FROM {catalog}.{schema_ops}.ingestion_requests
    WHERE load_id = '{load_id}' AND entity_name = '{entity_name}'
""").first()[0]

spark.createDataFrame([{
    "load_id": load_id, "entity_name": entity_name,
    "records_extracted": num_records_extracted, "records_ingested": num_records_extracted,
    "api_calls_count": api_calls,
    "checkpoint_before": str(checkpoint_value) if checkpoint_value else None,
    "checkpoint_after": str(max_id) if max_id else None,
    "execution_time_sec": ingest_total, "status": "SUCCESS" if ingest_success else "FAILED",
    "error_message": ingest_error, "run_date": run_date, "env": env, "created_at": datetime.now()
}]).write.mode("append").saveAsTable(f"{catalog}.{schema_log}.bronze_ingest_runs")

print(f"✅ Log registrado: {num_records_extracted} registros, {api_calls} calls, {ingest_total}s")
print("="*70)

## ✅ Bloco 10: Validação

In [0]:
print("✅ Validando...\n")
print("="*70)

total = spark.sql(f"SELECT COUNT(*) FROM {table_bronze}").first()[0]
this_load = spark.sql(f"SELECT COUNT(*) FROM {table_bronze} WHERE load_id = '{load_id}'").first()[0]

print(f"📊 VALIDAÇÃO 1: Total = {total:,}")
print(f"📊 VALIDAÇÃO 2: Este load = {this_load:,} (esperado {num_records_extracted:,})")
print(f"   {'✅ OK' if this_load == num_records_extracted else '⚠️ DIVERGÊNCIA'}\n")

print("📊 VALIDAÇÃO 3: Amostra\n")
display(spark.sql(f"""
    SELECT record_id, LEFT(record_hash, 12) as hash, ingestion_timestamp, ingestion_date, env, LEFT(payload, 80) as payload_preview
    FROM {table_bronze} WHERE load_id = '{load_id}' ORDER BY ingestion_timestamp LIMIT 5
"""))

print("\n📊 VALIDAÇÃO 4: Por Partição\n")
display(spark.sql(f"""
    SELECT ingestion_date, COUNT(*) as records, COUNT(DISTINCT load_id) as loads
    FROM {table_bronze} GROUP BY ingestion_date ORDER BY ingestion_date DESC LIMIT 10
"""))

print("\n" + "="*70)
print(f"\n🎯 RESUMO: {table_bronze}")
print(f"   • Total: {total:,} | Novos: {this_load:,} | Checkpoint: {max_id} | Status: SUCCESS ✅")
print("\n🎉 Notebook 04_bronze_votacoes concluído!")